# Team Ops RAG in Colab (Ollama + Drive ZIP)

This notebook runs fully in **Google Colab** with a local Ollama model and team documents from Google Drive:
- ZIP source: `/content/drive/MyDrive/Operations.zip`

We will:
1. Set up Ollama in Colab
2. Mount Drive and extract your team data
3. Build a RAG index
4. Ask baseline and advanced retrieval questions

## First Step: Enable Free GPU

Before running anything else:
- Runtime -> Change runtime type
- Hardware accelerator -> `T4 GPU`
- Click `Save`

## 1) Setup Ollama

In [ ]:
# Install Python dependencies
%pip install -q jedi pypdf llama-index llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
# Install Ollama + required system dependency
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama server (safe to rerun)
import os
import time
import subprocess

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

check = subprocess.run(["bash", "-lc", "pgrep -f 'ollama serve'"], capture_output=True, text=True)
if check.returncode != 0:
    _ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(4)
    print("Started Ollama server.")
else:
    print("Ollama server is already running.")

!ollama list

In [ ]:
# Pull models (small enough for Colab)
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text

## 2) Load Team Data From Drive ZIP

We will mount Drive, check for `Operations.zip`, extract it, and inspect files before indexing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Locate ZIP and extract
import os
import zipfile
from pathlib import Path

zip_path = Path('/content/drive/MyDrive/Operations.zip')
extract_dir = Path('/content/team_ops_data')

if not zip_path.exists():
    raise FileNotFoundError(
        f"Could not find {zip_path}. Please check the file name/path in Drive."
    )

extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print(f"Extracted ZIP to: {extract_dir}")

In [ ]:
# Quick file inventory
from pathlib import Path

all_files = [p for p in Path('/content/team_ops_data').rglob('*') if p.is_file()]
print(f"Total files found: {len(all_files)}")
for p in all_files[:30]:
    print('-', p)
if len(all_files) > 30:
    print('...')

## 3) Configure LLM, Embeddings, and Build RAG Index

- LLM: `llama3.2:3b` (generation)
- Embeddings: `nomic-embed-text` (vector retrieval)

In [ ]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model='llama3.2:3b', request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name='nomic-embed-text')
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print('LlamaIndex settings configured.')

In [ ]:
# Load documents and build vector index
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

data_dir = '/content/team_ops_data'
documents = SimpleDirectoryReader(data_dir, recursive=True).load_data()
if len(documents) == 0:
    raise ValueError('No readable documents loaded. Check file types and folder content.')

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print(f'Loaded {len(documents)} documents and built index.')

## 4) Compare: Plain LLM vs RAG

In [ ]:
# Plain LLM (no RAG)
question = 'How do I submit my timesheets?'
plain_response = Settings.llm.complete(question)
print('=== Plain LLM (no retrieval) ===')
print(getattr(plain_response, 'text', str(plain_response)))

In [ ]:
# RAG response
rag_response = query_engine.query(question)
print('=== RAG response ===')
print(rag_response)

## 5) Advanced Retrieval (Reranking)

Reranking often improves relevance by filtering a wider retrieved candidate set down to the best chunks.

In [ ]:
from llama_index.core.postprocessor import LLMRerank

reranker = LLMRerank(choice_batch_size=4, top_n=3)
rerank_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
)

rerank_response = rerank_engine.query(question)
print('=== RAG + Rerank response ===')
print(rerank_response)

In [ ]:
print('--- Baseline source nodes ---')
for i, node in enumerate(rag_response.source_nodes, 1):
    print(f'[{i}] score={node.score:.4f} | {node.text[:120]}...')

print()
print('--- Reranked source nodes ---')
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f'[{i}] score={node.score:.4f} | {node.text[:120]}...')

In [ ]:
# Ask your own question
your_question = input('Ask a question about your Ops docs: ')
print(rerank_engine.query(your_question))